# Augmentation Pipeline

This notebook prepares the dataset split and then augments only the training side.

- Source images are read from `data/raw`.
- Split output is written to `data/train` and `data/test`.
- Augmentation is applied only to `data/train`.

Run this notebook first.

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Optional
from collections import OrderedDict
import random
import shutil

import cv2
import numpy as np
from numpy.typing import NDArray
from PIL import Image

ImageArray = NDArray[np.uint8]

try:
    from pillow_heif import register_heif_opener
    register_heif_opener()
    HEIF_ENABLED: bool = True
except Exception:
    HEIF_ENABLED = False

random.seed(42)
np.random.seed(42)

project_root: Path = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent

RAW_FACE_DIR: Path = project_root / "data" / "raw"
TRAIN_DIR: Path = project_root / "data" / "train"
TEST_DIR: Path = project_root / "data" / "test"
IMAGE_EXTS: set[str] = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".heic", ".heif"}
AUGMENT_PER_IMAGE: int = 5
JPEG_QUALITY: int = 95
MAX_EDGE: int = 1600
TEST_RATIO: float = 0.2
SPLIT_SEED: int = 42

print(f"Raw dataset folder  : {RAW_FACE_DIR}")
print(f"Train dataset folder: {TRAIN_DIR}")
print(f"Test dataset folder : {TEST_DIR}")
print(f"HEIF enabled        : {HEIF_ENABLED}")

Face folder: c:\Users\harry\Documents\school\ip\AttSystem\data\face
HEIF enabled: True


In [ ]:
raw_dir = RAW_FACE_DIR

if not raw_dir.exists() or not raw_dir.is_dir():
    raise FileNotFoundError(
        f"Could not find {raw_dir}. Please provide the dataset in data/raw."
    )

counts = OrderedDict()
for folder in sorted([p for p in raw_dir.iterdir() if p.is_dir()]):
    num_images = sum(1 for f in folder.iterdir() if f.is_file() and f.suffix.lower() in IMAGE_EXTS)
    counts[folder.name] = num_images

target = 30

print(f"Dataset folder: {raw_dir}")
print("=" * 40)
for name, count in counts.items():
    remaining = max(0, target - count)
    status = "done" if remaining == 0 else f"need {remaining} more"
    print(f"{name:<15} : {count:>3} images | {status}")

print("=" * 40)
print(f"Total folders: {len(counts)}")
print(f"Total images : {sum(counts.values())}")

Dataset folder: c:\Users\harry\Documents\school\ip\AttSystem\data\face
benjamin        : 180 images | done
chern_tak       : 168 images | done
chillien        :  54 images | done
daniel          : 180 images | done
dylan           : 126 images | done
han_soon        : 120 images | done
harry           :  60 images | done
isaac           :  60 images | done
jing_ang        : 180 images | done
jun_wei         : 162 images | done
kang_kai        : 180 images | done
marion          : 168 images | done
ms_nurul        :  66 images | done
qi_xuan         : 180 images | done
shuang_quan     : 270 images | done
wee_xuan        : 144 images | done
xiang_yue       : 180 images | done
xu_sheng        : 174 images | done
yoke_hong       : 192 images | done
yong_kang       : 144 images | done
zi_herng        :  60 images | done
Total folders: 21
Total images : 3048


In [7]:
def load_image(file_path: Path) -> Optional[ImageArray]:
    suffix: str = file_path.suffix.lower()

    if suffix in {".heic", ".heif"}:
        if not HEIF_ENABLED:
            return None
        try:
            with Image.open(file_path) as im:
                rgb = im.convert("RGB")
                arr = np.array(rgb, dtype=np.uint8)
                return cv2.cvtColor(arr, cv2.COLOR_RGB2BGR) # type: ignore
        except Exception:
            return None

    img = cv2.imread(str(file_path), cv2.IMREAD_COLOR)
    if img is None:
        return None
    return img.astype(np.uint8, copy=False)


def resize_if_large(image: ImageArray, max_edge: int = MAX_EDGE) -> ImageArray:
    h, w = image.shape[:2]
    current_max: int = max(h, w)
    if current_max <= max_edge:
        return image
    scale: float = max_edge / float(current_max)
    new_w: int = int(w * scale)
    new_h: int = int(h * scale)
    resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA)
    return resized.astype(np.uint8, copy=False)


def aug_quality(image: ImageArray) -> ImageArray:
    alpha: float = random.uniform(0.75, 1.25)
    beta: float = random.uniform(-25, 25)
    out = cv2.convertScaleAbs(image, alpha=alpha, beta=beta)

    if random.random() < 0.5:
        out = cv2.GaussianBlur(out, (3, 3), 0)

    if random.random() < 0.6:
        noise = np.random.normal(0, random.uniform(5, 12), out.shape).astype(np.int16)
        out = np.clip(out.astype(np.int16) + noise, 0, 255).astype(np.uint8)

    if random.random() < 0.5:
        gamma: float = random.uniform(0.8, 1.3)
        table = np.array([((i / 255.0) ** (1.0 / gamma)) * 255 for i in np.arange(256)], dtype=np.uint8)
        out = cv2.LUT(out, table)

    return out.astype(np.uint8, copy=False)


def aug_angle(image: ImageArray) -> ImageArray:
    h, w = image.shape[:2]
    center: tuple[int, int] = (w // 2, h // 2)

    angle: float = random.uniform(-18, 18)
    scale: float = random.uniform(0.95, 1.05)
    mat = cv2.getRotationMatrix2D(center, angle, scale)
    out = cv2.warpAffine(image, mat, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)

    if random.random() < 0.5:
        tx: int = int(random.uniform(-0.04, 0.04) * w)
        ty: int = int(random.uniform(-0.04, 0.04) * h)
        tmat = np.array([[1.0, 0.0, float(tx)], [0.0, 1.0, float(ty)]], dtype=np.float32)
        out = cv2.warpAffine(out, tmat, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)

    return out.astype(np.uint8, copy=False)


def generate_augmented(image: ImageArray, total: int = AUGMENT_PER_IMAGE) -> list[ImageArray]:
    variants: list[ImageArray] = []
    for _ in range(total):
        out = image.copy()
        out = aug_angle(out)
        out = aug_quality(out)
        variants.append(out)
    return variants

In [ ]:
def split_raw_face_dataset(
    raw_face_dir: Path,
    train_dir: Path,
    test_dir: Path,
    test_ratio: float = TEST_RATIO,
    seed: int = SPLIT_SEED,
    overwrite: bool = True,
) -> None:
    if not raw_face_dir.exists() or not raw_face_dir.is_dir():
        raise FileNotFoundError(f"Missing raw dataset folder: {raw_face_dir}")

    if overwrite:
        shutil.rmtree(train_dir, ignore_errors=True)
        shutil.rmtree(test_dir, ignore_errors=True)

    train_dir.mkdir(parents=True, exist_ok=True)
    test_dir.mkdir(parents=True, exist_ok=True)

    rng = random.Random(seed)
    people_processed = 0

    for person_dir in sorted([p for p in raw_face_dir.iterdir() if p.is_dir()]):
        files = sorted([f for f in person_dir.iterdir() if f.is_file() and f.suffix.lower() in IMAGE_EXTS])
        if not files:
            continue

        names = [f.name for f in files]
        rng.shuffle(names)

        if len(names) <= 1:
            test_names = set()
        else:
            test_count = int(round(len(names) * test_ratio))
            test_count = max(1, min(test_count, len(names) - 1))
            test_names = set(names[:test_count])

        person_train_dir = train_dir / person_dir.name
        person_test_dir = test_dir / person_dir.name
        person_train_dir.mkdir(parents=True, exist_ok=True)
        person_test_dir.mkdir(parents=True, exist_ok=True)

        train_count = 0
        test_count = 0
        for file_path in files:
            target_dir = person_test_dir if file_path.name in test_names else person_train_dir
            shutil.copy2(file_path, target_dir / file_path.name)
            if target_dir is person_test_dir:
                test_count += 1
            else:
                train_count += 1

        people_processed += 1
        print(f"{person_dir.name}: train={train_count}, test={test_count}, total={len(files)}")

    print("\nDataset split complete")
    print(f"People processed: {people_processed}")
    print(f"Train folder: {train_dir}")
    print(f"Test folder : {test_dir}")


def standardize_and_augment_face_folder(face_dir: Path) -> None:
    if not face_dir.exists() or not face_dir.is_dir():
        raise FileNotFoundError(f"Missing dataset folder: {face_dir}")

    person_dirs: list[Path] = sorted([p for p in face_dir.iterdir() if p.is_dir()])
    if not person_dirs:
        raise RuntimeError(f"No people folders found in: {face_dir}")

    total_people: int = 0
    total_original: int = 0
    total_augmented: int = 0

    for person_dir in person_dirs:
        files: list[Path] = sorted([f for f in person_dir.iterdir() if f.is_file() and f.suffix.lower() in IMAGE_EXTS])
        if not files:
            continue

        loaded: list[ImageArray] = []
        for file_path in files:
            img = load_image(file_path)
            if img is None:
                continue
            img = resize_if_large(img)
            loaded.append(img)

        if not loaded:
            continue

        # Reset image files so naming is deterministic every run.
        for old_file in person_dir.iterdir():
            if old_file.is_file() and old_file.suffix.lower() in IMAGE_EXTS:
                old_file.unlink(missing_ok=True)

        person_name: str = person_dir.name
        original_count: int = 0
        aug_count: int = 0

        for idx, img in enumerate(loaded, start=1):
            base_name: str = f"{person_name}_{idx:04d}"
            original_path: Path = person_dir / f"{base_name}.jpg"
            cv2.imwrite(str(original_path), img, [int(cv2.IMWRITE_JPEG_QUALITY), JPEG_QUALITY])
            original_count += 1

            variants: list[ImageArray] = generate_augmented(img, total=AUGMENT_PER_IMAGE)
            for a_idx, var in enumerate(variants, start=1):
                aug_path: Path = person_dir / f"{base_name}_aug_{a_idx:02d}.jpg"
                cv2.imwrite(str(aug_path), var, [int(cv2.IMWRITE_JPEG_QUALITY), JPEG_QUALITY])
                aug_count += 1

        total_people += 1
        total_original += original_count
        total_augmented += aug_count
        print(f"{person_name}: originals={original_count}, augmented={aug_count}, total={original_count + aug_count}")

    print("\nPreprocessing finished")
    print(f"People processed: {total_people}")
    print(f"Standardized originals: {total_original}")
    print(f"Augmented images: {total_augmented}")
    print(f"Grand total in train/: {total_original + total_augmented}")


split_raw_face_dataset(RAW_FACE_DIR, TRAIN_DIR, TEST_DIR, test_ratio=TEST_RATIO, seed=SPLIT_SEED, overwrite=True)
standardize_and_augment_face_folder(TRAIN_DIR)

benjamin: originals=180, augmented=900, total=1080
chern_tak: originals=168, augmented=840, total=1008
chillien: originals=54, augmented=270, total=324
daniel: originals=180, augmented=900, total=1080
dylan: originals=126, augmented=630, total=756
han_soon: originals=120, augmented=600, total=720
harry: originals=60, augmented=300, total=360
isaac: originals=60, augmented=300, total=360
jing_ang: originals=180, augmented=900, total=1080
jun_wei: originals=162, augmented=810, total=972
kang_kai: originals=180, augmented=900, total=1080
marion: originals=168, augmented=840, total=1008
ms_nurul: originals=66, augmented=330, total=396
qi_xuan: originals=180, augmented=900, total=1080
shuang_quan: originals=270, augmented=1350, total=1620
wee_xuan: originals=144, augmented=720, total=864
xiang_yue: originals=180, augmented=900, total=1080
xu_sheng: originals=174, augmented=870, total=1044
yoke_hong: originals=192, augmented=960, total=1152
yong_kang: originals=144, augmented=720, total=864
